# Session 1: What a machine means when it says it learns

**259.075 Data-integrated Algorithmic Design Processes II** · Session 1 of 7

How much heating does a building need? Working it out properly takes an afternoon of simulation.
Today you build a **model**, a simple rule that predicts one number from another, and let it learn
the heating demand of 1073 Vienna buildings. The rule inside it will be found by a few lines that
**you** wrote. One honest warning up front: our heating demand is itself computed, not measured;
the box below says why.

We go **by foot**. First four buildings, in plain Python, every number printed, so you see each
computation happen. Then numpy and pandas, as shortcuts for what you have just done by hand. No
maths beyond adding, subtracting, multiplying and dividing.

Before most cells you are asked to **guess first**: say or write down what you expect, then run the
cell and compare. The answers are in the grey boxes under the cells; open them after guessing.

Timing: Part 1 (steps 1 to 6) about 40 minutes, Part 2 (steps 7 to 11) about 35. In Part 2 the
code is mostly supplied: read the comments where you are curious, but spend your time on the
questions and on your one line in step 10. Activities marked *optional* (the map, the animation)
are extras: when time is short, still run their cells with Shift + Enter, but skip the activity.

<div style="border-left: 5px solid #d8a24a; background-color: #fdf6e3; color: #222; padding: 10px 16px; margin: 6px 0;">

**💡 About the data**

The buildings and their construction periods come from the open data of the City of Vienna. The
other columns are ours: area, perimeter and compactness are measured from each outline, the
U-value of the wall is a textbook value per construction period, and the heating demand is
**computed** from those two, because no open data gives measured consumption per building. Keep
that in mind; it matters at the end.

<details>
<summary>What the columns mean. Click to open.</summary>

| column | meaning |
|---|---|
| `id` | the city's number for the building |
| `construction_period` | as the city records it; *Vor* means before, *Nach* after |
| `start_year` | a year taken from the period, used for colouring |
| `building_type` | the city's code for the type of building |
| `area_m2` | footprint area in m² |
| `perimeter_m` | perimeter of the footprint in m |
| `compactness` | see step 1 |
| `u_value` | U-value of the wall in W/m²K: how much heat passes through it; higher means leakier |
| `heating_demand` | in kWh per m² and year; the computed column |
| `lat`, `lon` | position on the map |

</details>

</div>

## Part 1: four buildings, by foot

### Step 1: one building, and its compactness

Click into the cell below and press **Shift + Enter**. That runs it. Do the same with every code
cell, from top to bottom.

The first building in our example is a small house from 1860 to 1883, id 27025801. Its footprint
has a perimeter of 25.4 m and an area of 40.3 m². Its **compactness** is the perimeter divided by
the square root of the area. The square root of 100 is 10, because 10 × 10 = 100, so a square house
of 10 × 10 m gets 40 m / 10 m = 4. The metres cancel: compactness has no unit. A circle gets about
3.5, the lowest possible; a long, thin slab gets much more. **A higher number means a less compact
shape.**

**Guess first:** this house is almost square. Roughly what compactness will it get?

In [ ]:
# A variable: the name perimeter now stands for the number 25.4
perimeter = 25.4
# Another variable, for the footprint area in m²
area = 40.3
# print shows values; text in quotes is printed as it is
print("perimeter:", perimeter, "m")
# Same kind: show the area
print("area:", area, "m²")

# "import math" loads Python's maths tools; math.sqrt is the square root
import math
# The square root of the area, kept under its own name
root = math.sqrt(area)
# Show the square root
print("square root of the area:", root)

# Compactness: perimeter divided by that square root (/ divides)
compactness_1 = perimeter / root
# The f before the quotes lets us put values into the text in {...};
# :.2f means "show two decimals"
print(f"compactness: {perimeter} / {root:.2f} = {compactness_1:.2f}")

<details>
<summary>Answer</summary>

4.00: 25.4 / 6.35. A perfect square gets exactly 4, so to two decimals this house is as compact as
a square.

</details>

One thing that confuses everyone once: a notebook remembers what you ran, **in the order you ran
it**, not in the order the cells stand on the page. If you change a cell above and do not run it
again, the cells below still use the old value. When in doubt: Kernel, Restart Kernel and Run All
Cells.

### Step 2: four buildings in two lists

Three more buildings from the table join the first. A **list** keeps several values in a row, in
square brackets. One list holds the compactness of the four, rounded to one decimal: 4.0, 5.0, 6.0
and 8.0. A second holds their heating demand in kWh/m²a, kilowatt-hours per square metre of floor
and year (*a* for *annum*), rounded to whole numbers: 147, 153, 159 and 183.

**Guess first:** Python counts positions from 0, so position 0 is the first value. What does
`demand[1]` print?

In [ ]:
# A list: four values in square brackets, separated by commas.
# The compactness of buildings 27025801, 25543683, 27026397, 23530759
compactness = [4.0, 5.0, 6.0, 8.0]
# Their heating demand in kWh/m²a, in the same order
demand = [147, 153, 159, 183]
# Show both lists
print("compactness:", compactness)
print("demand:     ", demand)

# Indexing: demand[0] is the FIRST value, because Python counts from 0
first = demand[0]
# Show it
print("demand[0] is", first)
# demand[1] is the second value
second = demand[1]
# Show it
print("demand[1] is", second)

# len(...) counts the values in a list
count = len(demand)
# Show it
print("how many buildings:", count)

# A for loop: the indented lines below it run once for every value
# in the list. Each time round, c is the next value.
for c in compactness:
    # This line is indented, so it belongs to the loop
    print("  one building with compactness", c)

<details>
<summary>Answer</summary>

153, the **second** value. `demand[0]` is 147, the first. Every list position in this course
counts from 0.

</details>

### Step 3: a line as a model

Our model is a straight line:

    predicted demand = w × compactness + b

`w` is the line's **slope**: how much the predicted demand rises when compactness goes up by one.
`b` lifts or lowers the whole line. With w = 8 and b = 110, the first building, compactness 4.0,
gets 8 × 4.0 + 110 = 142 kWh/m²a. A 100 m² flat at 142 kWh/m²a would need 14 200 kWh of heating a
year.

In the language of the field, this line is a **neuron**: an input (compactness), a **weight**
(`w`), a **bias** (`b`), an output. Every network later in this course is built from neurons like
this one. The cell writes the line as a **function**, a named recipe you can call again and again.

**Guess first:** what does the line predict for the other three buildings?

In [ ]:
# def makes a function called predict, with three inputs: the
# compactness c, the weight w and the bias b. Nothing runs yet.
def predict(c, w, b):
    # return hands the result back to whoever called the function.
    # * multiplies, + adds.
    return w * c + b

# A first guess for the weight
w = 8
# ... and for the bias
b = 110

# An empty list, to collect the predictions
predictions = []
# One prediction per building
for c in compactness:
    # Call the function: c, w and b go in, the prediction comes out
    p = predict(c, w, b)
    # append adds one value at the end of the list
    predictions.append(p)
    # Show the arithmetic
    print(f"compactness {c}: 8 × {c} + 110 = {p}")

# The whole list, after the loop
print("all predictions:", predictions)

<details>
<summary>Answer</summary>

8 × 5.0 + 110 = 150, 8 × 6.0 + 110 = 158 and 8 × 8.0 + 110 = 174. The table says 147, 153, 159
and 183: all four predictions are too low.

</details>

### Step 4: one number for "how wrong"

To find a good line, we need one number that says how wrong a line is. For each building: the
**miss** (predicted minus table), then the miss **squared**, the miss times itself. Squaring makes
every miss positive, so misses above and below do not cancel out, and it makes one big miss count
more than several small ones. Then all squares are added up and divided by how many.

> This average of the squared misses is the **mean squared error**, short **MSE**. The smaller,
> the better the line.

The loop keeps a **running total**: it starts at 0 and grows by one square per building. The first
building: predicted 142, table 147, miss −5, squared 25. **Guess first:** what is the MSE of the
line w = 8, b = 110 on all four?

In [ ]:
# The running total of squared misses starts at 0
total = 0
# range(4) gives the positions 0, 1, 2, 3: one per building
for n in range(4):
    # The prediction for building n, with w = 8 and b = 110
    p = predict(compactness[n], 8, 110)
    # The miss: predicted minus table (- subtracts)
    miss = p - demand[n]
    # The miss squared: the miss times itself
    square = miss * miss
    # The total grows by this square. The right side is worked out
    # first, with the old total; then the name total gets the result.
    total = total + square
    # Show this building's numbers
    print(f"building {n}: miss {miss}, squared {square}, total {total}")

# The mean: the total divided by how many buildings (count, from step 2)
mse_by_hand = total / count
# Show the division
print(f"MSE = {total} / {count} = {mse_by_hand}")

<details>
<summary>Answer</summary>

The misses are −5, −3, −1 and −9. Their squares are 25, 9, 1 and 81, together 116, divided by 4: **29**.

</details>

We need the same computation for other lines, so the next cell packs exactly this loop into a
function, `mse(w, b, xs, ys)`: a line and two lists in, the MSE out. Nothing else is new.

In [ ]:
# The MSE of the line w * c + b, for the inputs xs and the table
# values ys (two lists of the same length)
def mse(w, b, xs, ys):
    # The running total starts at 0
    total = 0
    # How many pairs there are
    count = len(xs)
    # One round per pair: positions 0, 1, ... up to count - 1
    for n in range(count):
        # The prediction for pair n
        p = predict(xs[n], w, b)
        # Its miss: predicted minus table
        miss = p - ys[n]
        # The miss squared
        square = miss * miss
        # Add the square to the running total
        total = total + square
    # Hand back the mean
    return total / count


# Check: the same line as above must give the same 29
check = mse(8, 110, compactness, demand)
# Show it
print("mse(8, 110) on the four buildings:", check)

### Step 5: which way is better? A tiny step

Should `w` go up or down? Make `w` a tiny bit bigger, 0.001, and see how the MSE changes. The change
divided by 0.001 is how fast the MSE changes per whole unit of `w`. This number is called the
**gradient**. Negative means "making `w` bigger makes the error smaller".

> To improve, step **against** the gradient: if it is negative, increase `w`; if positive,
> decrease it. Repeating that is called **gradient descent**.

The cell does the same for `b`: nudge `b`, hold `w` still.

**Guess first:** all four predictions were too low. Will the gradient for `w` be positive
or negative?

In [ ]:
# The size of the tiny step
tiny = 0.001
# The MSE where we stand: w = 8, b = 110
here = mse(8, 110, compactness, demand)
# The MSE with w a tiny step bigger, b unchanged
further = mse(8 + tiny, 110, compactness, demand)
# How much the MSE changed
change = further - here
# Change per whole unit of w: the gradient for w
gradient_w = change / tiny

# :.6f shows six decimals, so the small change is visible
print(f"MSE here:            {here:.6f}")
print(f"MSE with w + 0.001:  {further:.6f}")
print(f"change:              {change:.6f}")
print(f"change / 0.001:      {gradient_w:.2f}")

# The same for b: a tiny step in b, w unchanged
further_b = mse(8, 110 + tiny, compactness, demand)
# How much the MSE changed
change_b = further_b - here
# Change per whole unit of b: the gradient for b
gradient_b = change_b / tiny
# Same kind of output as for w
print(f"MSE with b + 0.001:  {further_b:.6f}")
print(f"change:              {change_b:.6f}")
print(f"change / 0.001:      {gradient_b:.2f}")

<details>
<summary>Answer</summary>

Negative, about −56.5: the MSE falls by about 0.056 when `w` grows by 0.001. So `w` should grow.
It is large because `w` is multiplied by compactness, 4 to 8: moving `w` moves every prediction a
lot. The gradient for `b` is −9.00, also negative but much smaller.

</details>

The next cell packs both into one function.
Both gradients are measured **before** either number changes; otherwise the second would be
measured on a line that has already moved.

In [ ]:
# Both gradients of the MSE at the line (w, b), each by a tiny step
def gradients(w, b, xs, ys):
    # The tiny step
    tiny = 0.001
    # The MSE where we stand
    here = mse(w, b, xs, ys)
    # A tiny step in w, b unchanged
    further_w = mse(w + tiny, b, xs, ys)
    # A tiny step in b, w unchanged
    further_b = mse(w, b + tiny, xs, ys)
    # The change for w
    change_w = further_w - here
    # The change per unit: the gradient for w
    g_w = change_w / tiny
    # The change for b
    change_b = further_b - here
    # The change per unit: the gradient for b
    g_b = change_b / tiny
    # Hand back two values at once, in this order: first g_w, then g_b
    return g_w, g_b


# Catch the two values in two names, in the same order
g_w, g_b = gradients(8, 110, compactness, demand)
# Show both gradients
print(f"gradient for w: {g_w:.2f}")
print(f"gradient for b: {g_b:.2f}")

### Step 6: two steps of the learning loop, with all the arithmetic

One learning step moves both numbers against their gradients:

    new w = w − learning rate × gradient for w
    new b = b − learning rate × gradient for b

The **learning rate** sets the step size. An example with small numbers: w = 10, gradient 4,
learning rate 0.1 gives 10 − 0.1 × 4 = 9.6. The minus makes the step go **against** the gradient,
downhill. Here the learning rate is 0.01.

**Guess first:** after the first step, will `w` be bigger or smaller than 8? Will the MSE be lower
than 29?

In [ ]:
# Start from our first guess, w = 8 ...
w = 8
# ... and b = 110
b = 110
# The size of the steps
learning_rate = 0.01

# How wrong the first guess is
start = mse(w, b, compactness, demand)
# Show where we start
print(f"start: w = {w}, b = {b}, MSE = {start:.2f}")
# Two rounds; step is 1, then 2
for step in [1, 2]:
    # Both gradients first, where we stand now
    g_w, g_b = gradients(w, b, compactness, demand)
    # Then both updates: minus, so each step goes against its gradient.
    # The new w:
    new_w = w - learning_rate * g_w
    # The new b, the same way:
    new_b = b - learning_rate * g_b
    # Show this step's arithmetic, w and b
    print(f"step {step}:")
    print(f"  w = {w:.3f} - {learning_rate} × ({g_w:.2f}) = {new_w:.3f}")
    print(f"  b = {b:.3f} - {learning_rate} × ({g_b:.2f}) = {new_b:.3f}")
    # The new numbers become the current ones: first w ...
    w = new_w
    # ... then b
    b = new_b
    # How wrong the moved line is
    now = mse(w, b, compactness, demand)
    # Show it
    print(f"  MSE now {now:.2f}")

<details>
<summary>Answer</summary>

Bigger: 8 − 0.01 × (−56.46) = 8.565, and `b` creeps from 110 to 110.090. The MSE drops from 29 to
8.12. The second step goes the same way, more gently: `w` 8.721, MSE 6.52. Notice that `b`
hardly moves: its gradient is small, because nudging `b` lifts every prediction by the same 0.001,
while nudging `w` lifts them by 4 to 8 times that.

</details>

That is all a machine does when it "learns": measure which way is downhill, take a small step, and
repeat. Part 2 does it thousands of times, on all 1073 buildings.

## Part 2: the same with numpy and pandas, then all buildings

### Step 7: numpy, as a shortcut

**numpy** gives you lists you can compute with in one go, called **arrays**. Three kinds of
shortcut appear in this course, each standing for a loop you could write yourself:

- **element by element**: `*`, `+`, `-`, `**` on an array work on every element separately;
- **weighted sums**: `@` multiplies and adds up a whole row at once (next session, with two
  inputs);
- **summaries**: `np.mean(...)` turns many numbers into one.

**Guess first:** a plain list is not an array. `[1, 2, 3] * 2` could multiply every value by 2,
or it could repeat the list twice. Which one does a plain list do?

In [ ]:
# "import numpy as np": load numpy and call it np for short
import numpy as np

# A plain list times 2: the list is REPEATED
doubled_list = [1, 2, 3] * 2
# Show it
print("list * 2: ", doubled_list)
# np.array(...) turns a list into an array
numbers = np.array([1, 2, 3])
# An array times 2: every ELEMENT is multiplied
doubled_array = numbers * 2
# Show it
print("array * 2:", doubled_array)

# Our four buildings as arrays
c = np.array(compactness)
# ... and the demand
d = np.array(demand)

# Element by element: all four predictions at once
p = 8 * c + 110
# Show it
print("predictions with numpy:", p)
# The list made by the loop in step 3
print("predictions by foot:   ", predictions)

<details>
<summary>Answer</summary>

`[1, 2, 3, 1, 2, 3]`: on a plain list, `*` repeats it. On an array it multiplies every element.
That is why the code turns lists into arrays before computing with them.

</details>

Now the MSE of step 4, as shortcuts. **Guess first:** will it be the same number as your loop?

In [ ]:
# Element by element: all four misses at once
misses = p - d
# Show them
print("misses:", misses)
# Element by element: each miss squared (** 2 means "to the power 2",
# the same as the miss times itself)
squares = misses ** 2
# Show them
print("squares:", squares)
# Summary: np.mean adds up and divides by how many
mse_with_numpy = np.mean(squares)
# Show both results, numpy and by foot
print("MSE with numpy:", mse_with_numpy)
print("MSE by foot:   ", mse_by_hand)

<details>
<summary>Answer</summary>

Yes, 29 both times. You will often see this in one line, `np.mean((w * x + b - y) ** 2)`: it is
the same three lines, predictions and misses, squares, mean. Read such lines from the inside out.

</details>

The next cell keeps this shortcut as a function, `mse_numpy`, the same recipe as `mse` from step 4
but without the loop.

In [ ]:
# The MSE with numpy: x and y are arrays, w and b single numbers
def mse_numpy(w, b, x, y):
    # Element by element: all predictions at once
    prediction = w * x + b
    # Element by element: all misses
    miss = prediction - y
    # Element by element: all squares
    square = miss ** 2
    # Summary: the mean of the squares
    return np.mean(square)


# Check against the loop, on another line, w = 7.5 and b = 120
by_loop = mse(7.5, 120, compactness, demand)
# The same line with numpy, on the arrays
by_numpy = mse_numpy(7.5, 120, c, d)
# Show both
print("loop: ", by_loop)
print("numpy:", by_numpy)

### Step 8: all 1073 buildings, with pandas

**pandas** reads a table from a file: one row per building, one column per property, each column
with a name. Two things to read in the cell: the path `"../data/..."` means "go one folder up, then
into data", and `df["compactness"]` picks one column by its name.

In [ ]:
# "import pandas as pd": load pandas and call it pd for short
import pandas as pd

# Read the whole table; df stands for "data frame", pandas' word for it
df = pd.read_csv("../data/buildings_vienna.csv")
# len(df) counts the rows
rows_count = len(df)
# A friendly check
print(f"✓ {rows_count} buildings loaded. Here are the first five rows:")
# head() shows the first five rows; the last line of a cell is displayed
df.head()

The next plot shows one dot per building: compactness across, heating demand up. The **U-value**
(`u_value`) says how much heat passes through a wall: the higher, the leakier. A wall with U = 1.50
lets twice as much heat through as one with U = 0.75. Run the cell and hover over a few dots.

In [ ]:
# plotly draws interactive plots; px is its quick version
import plotly.express as px
# go builds plots piece by piece
import plotly.graph_objects as go

# One dot per building, coloured by the start of its construction period
fig = px.scatter(df, x="compactness", y="heating_demand",
                 color="start_year",
                 hover_data=["id", "construction_period", "u_value"],
                 labels={"heating_demand": "heating demand [kWh/m2a]",
                         "start_year": "period from"},
                 color_continuous_scale="Viridis", height=500)
# Dot size and transparency
fig.update_traces(marker=dict(size=7, opacity=0.75))
fig.show()

Find two dots with almost the same compactness but clearly different heights, and compare their
`u_value` in the hover box. Why is their heating demand different?

<details>
<summary>Answer</summary>

Their U-values differ. Two houses of the same shape with different walls need different amounts of
heating. That is why the dots sit in several bands, one per U-value, instead of on one line.

</details>

**Optional: find a building you know.** The same buildings on a map, in four parts of Vienna. Look
for one you know, where you live, the institute, a corner you pass every day, and hover over it. Its
`id` appears in the box under the map; press **Copy id** and paste it into the cell after the map.
At the end you see how the model did on it. Give it five minutes at most.

In [ ]:
# HTML and display show our own HTML (here: the map and a box) in the notebook
from IPython.display import HTML, display

# Dots on a street map: colour = heating demand, size = footprint area
fig = px.scatter_map(df, lat="lat", lon="lon", color="heating_demand",
                     size="area_m2", color_continuous_scale="Turbo",
                     zoom=11, height=500, map_style="carto-positron",
                     hover_data=["id", "heating_demand",
                                 "construction_period"])

# A small box under the map, in JavaScript: hovering over a dot puts its
# id there, the button copies it
hover_box = """
var gd = document.getElementById('{plot_id}');
var box = document.createElement('div');
box.style.cssText = 'margin:8px 0;padding:8px 12px;border:1px solid #ccc;' +
    'border-radius:6px;font:14px sans-serif;display:flex;gap:10px;' +
    'align-items:center;';
box.innerHTML = '<span>Building under the mouse:</span>' +
    '<input readonly size="12" style="font:14px monospace;padding:3px">' +
    '<span class="info" style="color:#555"></span>' +
    '<button style="padding:4px 10px;cursor:pointer">Copy id</button>' +
    '<span class="ok" style="color:green"></span>';
gd.parentNode.insertBefore(box, gd.nextSibling);
var field = box.querySelector('input');
var info = box.querySelector('.info');
var ok = box.querySelector('.ok');
gd.on('plotly_hover', function (ev) {
    var d = ev.points[0].customdata;
    field.value = d[0];
    info.textContent = 'heating demand ' + Math.round(d[1]) +
        ' kWh/m²a, ' + d[2];
    ok.textContent = '';
});
box.querySelector('button').onclick = function () {
    if (!field.value) {
        ok.textContent = 'hover over a dot first';
        return;
    }
    var done = function () {
        ok.textContent = '✓ copied ' + field.value + ', now paste it below';
    };
    if (navigator.clipboard && window.isSecureContext) {
        navigator.clipboard.writeText(field.value).then(done);
    } else {
        field.select();
        document.execCommand('copy');
        done();
    }
};
"""
# The map as HTML, with the box script attached after it
page = fig.to_html(include_plotlyjs="cdn", full_html=False,
                   post_script=hover_box, config={"doubleClick": False})
html = HTML(page)
display(html)

In [ ]:
# YOUR CHOICE (optional): paste the id of your building here, instead of None
my_id = None

# All ids of the table, as a plain list
all_ids = df["id"].tolist()
# is None: nothing pasted yet
if my_id is None:
    # A reminder
    print("No building chosen. That is fine; paste an id to try it.")
# elif means "otherwise, if"; not in: the number is not among the ids
elif my_id not in all_ids:
    # A hint
    print(f"There is no building with id {my_id}. Check the number.")
# else: in every other case, found it
else:
    # == compares: True for the one row whose id equals my_id
    is_mine = df["id"] == my_id
    # df[is_mine] keeps only that row; .iloc[0] takes it out of the table
    my_house = df[is_mine].iloc[0]
    # Its details, as sentences
    print(f"✓ Your building {my_id}: built {my_house['construction_period']},")
    print(f"  compactness {my_house['compactness']:.2f}, "
          f"heating demand {my_house['heating_demand']:.0f} kWh/m²a.")
    print("  We come back to it at the end.")

The loop from step 4 works on all buildings too, only slower. The cell takes the two columns out of
the table as arrays and computes the MSE of the line w = 8, b = 110 both ways. **Guess first:**
the same number?

In [ ]:
# The two columns, as numpy arrays: x is compactness, y heating demand
x = df["compactness"].to_numpy()
# ... and y heating demand
y = df["heating_demand"].to_numpy()

# By foot: the loop from step 4, over all 1073 buildings
slow = mse(8, 110, x, y)
# Shortcut: numpy
fast = mse_numpy(8, 110, x, y)
# Show both
print(f"loop:  MSE {slow:.4f}")
print(f"numpy: MSE {fast:.4f}")

<details>
<summary>Answer</summary>

The same MSE, 53.39. The loop walks through 1073 buildings one by one; numpy says the same
in three lines and is faster on long tables. That matters, because the learning loop below needs
the MSE four times per step, for thousands of steps.

</details>

### Step 9: your line, by hand

Run the cell below and drag the two sliders under the plot, one for `w` and one for `b`. The red
line follows, and the box shows the MSE of that line on all 1073 buildings, computed with the same
loop as `mse`, in the browser. Find the smallest MSE you can and write it down.

In [ ]:
# json turns Python lists into text the browser can read
import json

# The two columns as text, for the browser
x_list_all = x.tolist()
x_text = json.dumps(x_list_all)
y_list_all = y.tolist()
y_text = json.dumps(y_list_all)
# The line only needs its two end points: smallest and largest compactness
ends = [x.min(), x.max()]
ends_text = json.dumps(ends)

# The cloud of buildings, and the red line for w = 8, b = 110
cloud = go.Scatter(x=x, y=y, mode="markers",
                   marker=dict(size=5, opacity=0.4, color="#4c78a8"))
line = go.Scatter(x=ends, y=[8 * ends[0] + 110, 8 * ends[1] + 110],
                  mode="lines", line=dict(width=4, color="crimson"))
fig = go.Figure([cloud, line])
# Fixed axes, so the line visibly moves
fig.update_layout(height=460, showlegend=False, yaxis_range=[40, 240],
                  xaxis_title="compactness",
                  yaxis_title="heating demand, computed")

# Two sliders under the plot, in JavaScript; moving them redraws the line
# and recomputes the MSE with a loop, like mse in step 4
sliders = """
var gd = document.getElementById('{plot_id}');
var xs = X_TEXT;
var ys = Y_TEXT;
var ends = ENDS_TEXT;
var box = document.createElement('div');
box.style.cssText = 'margin:8px 0 8px 80px;font:14px sans-serif;' +
    'display:grid;grid-template-columns:90px 260px 50px;gap:6px 12px;' +
    'align-items:center;';
box.innerHTML = '<span>w (slope)</span>' +
    '<input type="range" min="0" max="20" step="0.5" value="8">' +
    '<b class="vw"></b>' +
    '<span>b (lift)</span>' +
    '<input type="range" min="50" max="170" step="1" value="110">' +
    '<b class="vb"></b>' +
    '<span></span><span class="out" style="grid-column:2 / span 2"></span>';
gd.parentNode.insertBefore(box, gd.nextSibling);
var r = box.querySelectorAll('input');
function mse(w, b) {
    var total = 0;
    for (var n = 0; n < xs.length; n++) {
        var miss = w * xs[n] + b - ys[n];
        total = total + miss * miss;
    }
    return total / xs.length;
}
function redraw() {
    var w = parseFloat(r[0].value);
    var b = parseFloat(r[1].value);
    box.querySelector('.vw').textContent = w;
    box.querySelector('.vb').textContent = b;
    box.querySelector('.out').textContent =
        'MSE on all buildings: ' + mse(w, b).toFixed(1);
    var line = [w * ends[0] + b, w * ends[1] + b];
    Plotly.restyle(gd, {y: [line]}, [1]);
}
r[0].oninput = redraw;
r[1].oninput = redraw;
redraw();
"""
# Put the data into the script
sliders = sliders.replace("X_TEXT", x_text)
sliders = sliders.replace("Y_TEXT", y_text)
sliders = sliders.replace("ENDS_TEXT", ends_text)
# Draw plot and sliders
page = fig.to_html(include_plotlyjs="cdn", full_html=False,
                   post_script=sliders)
html = HTML(page)
display(html)

<details>
<summary>Answer</summary>

w = 8 with b = 110 gives an MSE of 53.4. The best the sliders allow is about 50.4, at
w = 8.5 and b = 109. Many lines look about right to the eye; the MSE tells them apart. Finding the line
with the smallest squared misses is over two hundred years old: Legendre and Gauss published it
around 1805 as the method of **least squares**.

</details>

### Step 10: your part, the learning loop

Now the computer does it by itself: thousands of the steps you did twice in step 6, on all 1073
buildings. The next cell gives the gradients a numpy version, exactly the function from step 5 with
`mse_numpy` in place of `mse`, and checks that both give the same.

In [ ]:
# The gradients of step 5, with the fast mse_numpy instead of the loop
def gradients_numpy(w, b, x, y):
    # The tiny step
    tiny = 0.001
    # The MSE where we stand
    here = mse_numpy(w, b, x, y)
    # A tiny step in w, b unchanged
    further_w = mse_numpy(w + tiny, b, x, y)
    # A tiny step in b, w unchanged
    further_b = mse_numpy(w, b + tiny, x, y)
    # The change for w
    change_w = further_w - here
    # The change per unit: the gradient for w
    g_w = change_w / tiny
    # The change for b
    change_b = further_b - here
    # The change per unit: the gradient for b
    g_b = change_b / tiny
    # Two values back: first g_w, then g_b
    return g_w, g_b


# Both versions, at w = 5 and b = 110, on all buildings
loop_w, loop_b = gradients(5, 110, x, y)
# The same with numpy
numpy_w, numpy_b = gradients_numpy(5, 110, x, y)
# Show both, side by side
print(f"✓ gradient for w: loop {loop_w:.4f}, numpy {numpy_w:.4f}")
print(f"✓ gradient for b: loop {loop_b:.4f}, numpy {numpy_b:.4f}")

The learning loop, as a function `fit`. It starts with a flat line at the average demand (`w` = 0,
`b` = the mean of `y`) and writes down `w`, `b` and the MSE after every step in a list called
`history`, one row per step.

One line is yours: the update of `b`. The update of `w` stands right above it as the example. In
step 6 you saw it with numbers; here it is with names.

Stuck? Ask your neighbour or the tutor right away, or open the answer below the cell.

In [ ]:
# The learning loop: x and y the data, then the step size and how many steps
def fit(x, y, learning_rate, steps):
    # Start: a flat line (slope 0) ...
    w = 0.0
    # ... at the average demand
    b = np.mean(y)
    # The history: a list of rows [w, b, MSE], the first row for the start
    start = mse_numpy(w, b, x, y)
    # The history: a list of rows [w, b, MSE], the first row the start
    history = [[w, b, start]]
    # range(steps) repeats the indented lines steps times
    for step in range(steps):
        # Both gradients first, where we stand now
        g_w, g_b = gradients_numpy(w, b, x, y)
        # Step w against its gradient
        w = w - learning_rate * g_w

        # Step b against its gradient
        b = b - learning_rate * g_b

        # How wrong the line is now
        now = mse_numpy(w, b, x, y)
        # One more row of the history
        history.append([w, b, now])
    # Hand back the whole history
    return history


# Run it: 3000 steps with learning rate 0.015
history = fit(x, y, 0.015, 3000)
# history[-1] is the LAST row: -1 counts from the end
last = history[-1]
# Position 0 of the row is w
w_end = last[0]
# Position 1 is b
b_end = last[1]
# Position 2 is the MSE
mse_end = last[2]
# The reward
print(f"✓ Your loop found the line by itself: w = {w_end:.2f}, "
      f"b = {b_end:.1f}, MSE = {mse_end:.1f}.")

<details>
<summary>Answer</summary>

    b = b - learning_rate * g_b

Your loop ends at an MSE of 50.3, at w = 8.49 and b = 108.9: as good as the best
slider setting and a hair better, because it can use any `w` and `b`, not only the slider
positions. And it found the line by itself.

</details>

The next cell prints a few rows of the history. Run it, then read: between which two printed
steps does the MSE first fall below 55?

In [ ]:
# A few stops along the way
for step in [0, 10, 30, 100, 300, 1000, 3000]:
    # The row of that step
    row = history[step]
    # Position 0 of the row is w
    w_at = row[0]
    # Position 1 is b
    b_at = row[1]
    # Position 2 is the MSE
    mse_at = row[2]
    # One line per stop
    print(f"step {step}: w = {w_at:.2f}, b = {b_at:.1f}, MSE = {mse_at:.1f}")

<details>
<summary>Answer</summary>

Between step 300 (MSE 94.4) and step 1000 (MSE 52.4). Most of the 3000 steps are spent on the last few points of improvement. Session 2
shows why it is so slow and how to make it fast.

</details>

**Optional: watch it learn.** Press **Play** below the plots. The left plot is a map of possible
lines: across is `w`, up is `b`, and the colour says how good that line is, dark is good. The white
star is the best line there is. The red path starts at the flat line, w = 0 and b = 155.4, with MSE
227.4. Notice the long, narrow, dark band running diagonally: a valley. The path walks down this
valley to the star, slowly, because its floor is almost flat. The right plot shows the line settling
into the cloud at the same time.

In [ ]:
# Two plots side by side
from plotly.subplots import make_subplots

# The MSE for a grid of 80 x 80 lines around the path, for the background
w_grid = np.linspace(-1, 11, 80)
b_grid = np.linspace(95, 165, 80)
rows = []
for b_value in b_grid:
    row = []
    for w_value in w_grid:
        value = mse_numpy(w_value, b_value, x, y)
        row.append(value)
    rows.append(row)
# The colour follows the logarithm of the MSE, so the valley shows
z = np.log10(rows)

# The history as an array: column 0 is w, column 1 b, column 2 the MSE
h = np.array(history)
# The best line, solved directly (np.polyfit, degree 1: a straight line)
best = np.polyfit(x, y, 1)
# Points along compactness, for drawing the line
x_low = x.min()
x_high = x.max()
xs = np.linspace(x_low, x_high, 50)

titles = ("path through the MSE landscape", "the line in the data")
fig = make_subplots(rows=1, cols=2, subplot_titles=titles)
fig.add_trace(go.Contour(x=w_grid, y=b_grid, z=z, colorscale="Viridis",
                         ncontours=25, showscale=False,
                         hovertemplate="w %{x:.2f}, b %{y:.1f}<extra></extra>"),
              row=1, col=1)
fig.add_trace(go.Scatter(x=[h[0, 0]], y=[h[0, 1]], mode="lines+markers",
                         marker=dict(size=7, color="crimson")),
              row=1, col=1)
fig.add_trace(go.Scatter(x=x, y=y, mode="markers",
                         marker=dict(size=5, opacity=0.4, color="#4c78a8")),
              row=1, col=2)
fig.add_trace(go.Scatter(x=xs, y=h[0, 0] * xs + h[0, 1], mode="lines",
                         line=dict(width=4, color="crimson")),
              row=1, col=2)
fig.add_trace(go.Scatter(x=[best[0]], y=[best[1]], mode="markers",
                         marker=dict(symbol="star", size=16, color="white",
                                     line=dict(width=1, color="black"))),
              row=1, col=1)

# Which steps to show: many at the start, where things change fast
shown = np.geomspace(1, len(history) - 1, 60)
shown = shown.astype(int)
shown = np.unique(shown)
frames = []
for i in shown:
    path = go.Scatter(x=h[:i + 1, 0], y=h[:i + 1, 1], mode="lines+markers",
                      marker=dict(size=5, color="crimson"))
    line = go.Scatter(x=xs, y=h[i, 0] * xs + h[i, 1], mode="lines",
                      line=dict(width=4, color="crimson"))
    title = f"step {i}, MSE {h[i, 2]:.1f}"
    frame = go.Frame(name=str(i), data=[path, line], traces=[1, 3],
                     layout=go.Layout(title=title))
    frames.append(frame)
fig.frames = frames

# Play and Stop buttons below the plots
play = dict(label="Play", method="animate",
            args=[None, dict(frame=dict(duration=80, redraw=True),
                             fromcurrent=True)])
stop = dict(label="Stop", method="animate",
            args=[[None], dict(mode="immediate")])
fig.update_layout(height=600, showlegend=False, title="step 0",
                  margin=dict(b=110),
                  updatemenus=[dict(type="buttons", direction="left",
                                    x=0, y=-0.2, xanchor="left",
                                    yanchor="top", buttons=[play, stop])])
fig.update_xaxes(title_text="w", row=1, col=1)
fig.update_yaxes(title_text="b", row=1, col=1)
fig.update_xaxes(title_text="compactness", row=1, col=2)
fig.update_yaxes(title_text="heating demand", row=1, col=2)
fig.show()

### Step 11: what did the model learn?

Your MSE stops at about 50, and the test below checks that this is close to the best
any straight line can do. That is not the loop's fault: the heating demand was computed from the
U-value **and** the compactness, but the line only sees the compactness.

The **residual** of a building is table minus predicted. If the table says 160 and the line says
150, the residual is +10: the line guessed too low. Note that this is the miss from step 4 with the
sign turned round. The map colours each building by its residual: blue guessed too low, red too
high.

In [ ]:
# What your line predicts for every building (element by element)
prediction = w_end * x + b_end
# Residual: table minus predicted, element by element
residual = y - prediction
# Both as new columns of the table, so the map can show them
df["prediction"] = prediction
df["residual"] = residual

# Map coloured by residual: blue = guessed too low, red = too high
fig = px.scatter_map(df, lat="lat", lon="lon", color="residual",
                     color_continuous_scale="RdBu", range_color=(-50, 50),
                     zoom=11, height=520, map_style="carto-positron",
                     hover_data=["id", "construction_period", "u_value",
                                 "prediction", "heating_demand"])
fig.show()

Hover over a few dots and compare their `u_value`. The next cell does it for all of them: for each
U-value, the average residual of the buildings with that wall. **Guess first:** the newest
buildings, after 1976, have the best walls (0.70). Will the line guess too high or too low for
them?

In [ ]:
# The five U-values that occur in the table
for u in [1.50, 1.45, 1.40, 1.30, 0.70]:
    # == compares: True for every building whose wall has this U-value.
    # (A single = gives a name a value; a double == asks "equal?")
    has_u = df["u_value"] == u
    # Keep only those rows of the table
    group = df[has_u]
    # Summary: the average residual of the group
    average = np.mean(group["residual"])
    # How many buildings have this wall
    size = len(group)
    # One line per U-value
    print(f"U-value {u:.2f}: {size} buildings, "
          f"average residual {average:+.1f}")

# Your own building, if you chose one in step 8: "is not None" means
# an id was pasted, "in all_ids" that it is in the table; "and" needs both
if my_id is not None and my_id in all_ids:
    # Its row of the table, as in step 8
    is_mine = df["id"] == my_id
    # .iloc[0] takes that one row out of the table
    mine = df[is_mine].iloc[0]
    # Its residual
    r = mine["residual"]
    # Which way the line was off: residual above 0 means too low
    if r > 0:
        # The line was below the table value
        direction = "too low"
    else:
        # Otherwise it was above
        direction = "too high"
    # Your result
    print(f"Your building {my_id}: U-value {mine['u_value']:.2f}, "
          f"residual {r:+.1f} kWh/m²a, so the line guessed {direction}.")

<details>
<summary>Answer</summary>

Too high: on average they get −44.3, far below everyone else. Buildings with 1.50 have
+4.9: the line guesses too low for them; 1.30 has −7.3. The line misses in a pattern that
follows the wall, and the wall comes from the construction period.

</details>

> Our model recovers a calculation rule that someone wrote down. It has not learned how a house
> behaves. Everything in this course stands or falls on that difference.

## Tests

First: Kernel, Restart Kernel and Run All Cells, so you know everything runs in order. Then this
cell recomputes: the hand MSE and `mse` must both give 116 / 4, the loop and numpy must agree, and your loop must end
within 5 percent of the smallest MSE any straight line can reach. If the best is 50, yours has to be
below 52.5.

In [ ]:
# assert checks something and stops with the message if it is false.
# abs(...) is the distance from 0; 1e-9 means 0.000000001, "equal up to
# tiny rounding"
assert abs(mse_by_hand - 116 / 4) < 1e-9, "the hand MSE should be 29"
# The function mse must give the same 29, computed again
again = mse(8, 110, compactness, demand)
# Must be 29
assert abs(again - 29) < 1e-9, "mse(8, 110) on the four should be 29"

# Loop and numpy must agree on all buildings, for another line
loop_value = mse(9, 100, x, y)
# ... and with numpy
numpy_value = mse_numpy(9, 100, x, y)
# Both must agree
assert abs(loop_value - numpy_value) < 1e-6, "loop and numpy disagree"

# Run your learning loop again, and take its last row
history_t = fit(x, y, 0.015, 3000)
# The last row of the history
last_t = history_t[-1]
# ... and the first
first_t = history_t[0]
# Its w
w_t = last_t[0]
# Its b
b_t = last_t[1]
# np.isfinite: False if w or b ran away to infinity
assert np.isfinite(w_t), "w ran away to infinity: check the minus sign"
# The same for b
assert np.isfinite(b_t), "b ran away to infinity: check the minus sign"
# The last MSE must be lower than the first
assert last_t[2] < first_t[2], "the MSE has to fall during the run"

# The best straight line, solved directly: np.polyfit with degree 1
best = np.polyfit(x, y, 1)
# Its MSE: best[0] is the slope, best[1] the bias
best_mse = mse_numpy(best[0], best[1], x, y)
# Your MSE
your_mse = mse_numpy(w_t, b_t, x, y)
# Yours within 5 percent of the best
assert your_mse < 1.05 * best_mse, (
    f"still too far off: {your_mse:.1f} against {best_mse:.1f} at best")
# Reached only if all checks pass
print(f"✓ All green. Your MSE {your_mse:.2f}, the best possible "
      f"{best_mse:.2f}.")

## Where you stand now

You computed by hand what a learning machine computes: predictions in a loop, how wrong a line is
as one number, which way is downhill by a tiny step, and the step itself. Then you let the loop do it
thousands of times, with numpy shortcuts you can now read: element by element (`*`, `-`, `**`) and
summaries (`np.mean`). And you know what your result does not mean: it reproduces a calculation
rule; it has not learned how a house behaves.

Next session the model gets the U-value as a second input.

---

### Terms to learn

These words are in the flashcard app, together with those of the other sessions, at
[gabriel-wurzer.github.io/data2/webapps/flashcards](https://gabriel-wurzer.github.io/data2/webapps/flashcards/).
Add it to your home screen; it works offline.

| term | in one sentence |
|---|---|
| model | a simplified rule that predicts an output from inputs; here, demand from compactness |
| neuron | inputs, one weight each, a bias, one output |
| weight, bias | the numbers the machine changes while it learns |
| parameter | the adjustable numbers of a model, such as `w` and `b` |
| loss | the general word for "how wrong the model is"; ours is the mean squared error |
| mean squared error (MSE) | the average of the squared misses |
| gradient | how much the loss changes per unit you move a parameter |
| gradient descent | repeatedly stepping against the gradient |
| learning rate | the number the gradient is multiplied by to give the step; too large and the loss rises |
| least squares | choosing the line whose squared misses add up to the least; Legendre and Gauss, around 1805 |
| residual | table (later: measured) minus predicted, for one building |

### Additional material

**Zeit im Bild**, a mock TV interview about AI:
[gabriel-wurzer.github.io/data2/webapps/zib](https://gabriel-wurzer.github.io/data2/webapps/zib/).
A presenter asks the questions everybody asks: will it take our jobs, does it understand what it
says, who is liable, is it all just statistics. Try answering before you look. Today you know what
a model is; by the end of the course you will be able to answer most of them properly, and we come
back to it in the last session.

## Homework: your own line

In class your loop found a line through 1073 Vienna buildings. Now give it something from your own
life. Pick two quantities that you suspect go together, collect 10 to 20 pairs of numbers, and let
your loop find the line. One of the pairs is a case you care about, for example your own flat.

Take numbers you can get quickly, from records you already have or can look up: rent and size of
flats on a listing site, distance and travel time of trips in your maps app, floor area and heating
costs of flats you know, length and plays of songs in your music app. Give the collecting about
twenty minutes.

The cells come with a complete worked example: invented but realistic rents of WG rooms in Vienna,
with example answers under the questions. Run it once first, read the example answers, then replace
the pairs with your own and write your own answers.

**How to do it**

1. Fill in the `YOUR TURN` gap above if you have not yet, yourself or from the solution; if from
   the solution, say so in one line at the top of your answers.
2. Below, change only the lines under `# YOUR CHOICE`, and write your answers into the last cell.
3. Choose Kernel, Restart Kernel and Run All Cells, and upload the notebook in TUWEL.

Before you pull updates of the course material, rename this notebook, for example to
`01_neuron_yourname.ipynb`, so your work is not overwritten.

**Points**

- Four for running it: your own pairs are in, the line is found, the plot is there.
- Six for your four answers about your own example. Answers that are about **your** numbers and say
  why get the points. There is no single right answer to hit, and "I cannot tell from my data,
  because ..." is a fine answer.

In [ ]:
# YOUR CHOICE: what you measured, with units
x_name = "room size"
# ... and its unit
x_unit = "m²"
# The second quantity
y_name = "rent"
# ... and its unit
y_unit = "€ per month"

# YOUR CHOICE: your pairs, one per line: first x, then y, separated by a
# semicolon. Decimal commas or points both work. Mark the one case you care
# about with a * at the end. The example: 14 WG rooms in Vienna, invented
# but realistic; the * is "my room". Replace it with your own pairs.
my_pairs = """
10; 390
12; 400
12,5; 450
14; 430
15; 470
16; 560 *
17; 490
18; 520
19; 500
20; 560
21; 540
22; 600
24; 610
26; 650
"""

# Two empty lists to fill, and no marked case yet
x_list = []
# ... one for y
y_list = []
# None: no marked case yet
my_point = None
# splitlines() cuts the text into a list of lines
lines = my_pairs.splitlines()
# Go through your text line by line
for line in lines:
    # strip() removes spaces at both ends
    text = line.strip()
    # An empty line: continue skips to the next line
    if text == "":
        # continue skips the rest of the loop for this line
        continue
    # Is this the case you care about? endswith checks the last character
    marked = text.endswith("*")
    # Remove the star, and turn decimal commas into points
    text = text.replace("*", "")
    # ... and turn decimal commas into points
    text = text.replace(",", ".")
    # split(";") cuts the line at the semicolon into a list of parts
    parts = text.split(";")
    # != means "not equal"; raise stops the cell with this message
    if len(parts) != 2:
        # Stop with a hint
        raise ValueError(f"this line needs exactly one semicolon: {line}")
    # try: attempt the indented lines; except: what to do if they fail
    try:
        # float(...) turns text such as "12.5" into the number 12.5
        x_value = float(parts[0])
        # ... and the y part
        y_value = float(parts[1])
    except ValueError:
        # from None: show only this message, not Python's own as well
        raise ValueError(f"this line has something that is not a number: "
                         f"{line}") from None
    # The marked case: remember its position (Python counts from 0)
    if marked:
        # A position is already stored: a second star
        if my_point is not None:
            # Stop with a hint
            raise ValueError("only one line may have a * at the end")
        # Its position: how many pairs came before it
        my_point = len(x_list)
    # Add the two numbers to their lists
    x_list.append(x_value)
    # ... and y to the list of y values
    y_list.append(y_value)

# Turn the lists into arrays, to compute with them
x_mine = np.array(x_list)
# ... and the y values
y_mine = np.array(y_list)
# Checks, each stops with its message if false. >= means "at least"
assert len(x_mine) >= 5, "at least five pairs, better 10 to 20"
# One line must be marked
assert my_point is not None, "mark the case you care about with a * at the end"
# np.isfinite(...).all(): every value is a real number (no nan or inf)
finite_x = np.isfinite(x_mine)
# .all(): True only if every value is
finite_x = finite_x.all()
# The same for y
finite_y = np.isfinite(y_mine)
# True only if every value is
finite_y = finite_y.all()
# Both must hold
assert finite_x and finite_y, "a value is nan or inf; type real numbers"
# The largest x must be bigger than the smallest
assert x_mine.max() > x_mine.min(), (
    "all your x values are the same, so these pairs cannot show how y "
    "changes with x; choose a quantity that varies")
# How many pairs
pair_count = len(x_mine)
# The x of your marked case
x_case = x_mine[my_point]
# ... and its y
y_case = y_mine[my_point]
# A short report
print(f"✓ {pair_count} pairs of {x_name} and {y_name}. Your case: "
      f"{x_case:g} {x_unit}, {y_case:g} {y_unit}.")

In [ ]:
# This cell changes the ruler while learning, then puts your units back: it
# shifts and stretches x so that any numbers (years, rents, minutes) walk
# equally fast. You do not need to follow it; the line you get is the same.
# The average of your x values, and their spread (np.std: how far they
# typically lie from the average)
x_mean = np.mean(x_mine)
# np.std: how far your x typically lie from their average
x_spread = np.std(x_mine)
# Your x on the new ruler: shifted, then stretched, element by element
x_shifted = x_mine - x_mean
# ... then stretched
x_ruler = x_shifted / x_spread
# Your own learning loop, with a bigger learning rate and 2000 steps
history_mine = fit(x_ruler, y_mine, 0.25, 2000)
# The last row of the history
last_mine = history_mine[-1]
# Position 0 is w (on the new ruler)
w_ruler = last_mine[0]
# Position 1 is b
b_ruler = last_mine[1]
# Position 2 is the MSE
mse_mine = last_mine[2]
# The slope and the lift, back in your units
w_mine = w_ruler / x_spread
# The lift back in your units
b_mine = b_ruler - w_mine * x_mean

# What your line says for your case (x_case from the cell above)
pred_mine = predict(x_case, w_mine, b_mine)
# Your residual: your value minus the line
res_mine = y_case - pred_mine
# The typical miss: the square root of the MSE, back in your y units
miss_typical = np.sqrt(mse_mine)

# The result, in four lines of the same kind
print(f"your line : {y_name} = {w_mine:.4g} × {x_name} + {b_mine:.4g}")
print(f"in words  : every extra {x_unit} of {x_name} goes with "
      f"{w_mine:+.4g} {y_unit} of {y_name}")
print(f"typical miss: {miss_typical:.4g} {y_unit}")
print(f"your case : line says {pred_mine:.4g} {y_unit}, "
      f"you have {y_case:g}, residual {res_mine:+.4g} {y_unit}")
# The smallest x of your data
x_min = x_mine.min()
# ... and the largest
x_max = x_mine.max()
# Same kind: the range of your data
print(f"your data : {x_name} from {x_min:g} to {x_max:g} {x_unit}")

In [ ]:
# Your pairs, your line, and your case in orange
ends = [x_mine.min(), x_mine.max()]
left = predict(ends[0], w_mine, b_mine)
right = predict(ends[1], w_mine, b_mine)
pairs = go.Scatter(x=x_mine, y=y_mine, mode="markers", name="your pairs",
                   marker=dict(size=10, color="#4c78a8"))
your_line = go.Scatter(x=ends, y=[left, right], mode="lines",
                       name="your line",
                       line=dict(width=4, color="crimson"))
your_case = go.Scatter(x=[x_mine[my_point]], y=[y_mine[my_point]],
                       mode="markers", name="your case",
                       marker=dict(size=16, color="orange",
                                   line=dict(width=2, color="black")))
fig = go.Figure([pairs, your_line, your_case])
fig.update_layout(height=450, xaxis_title=f"{x_name} [{x_unit}]",
                  yaxis_title=f"{y_name} [{y_unit}]",
                  title=f"your line: {y_name} = {w_mine:.4g} × {x_name} "
                        f"+ {b_mine:.4g}")
fig.show()

In [ ]:
# YOUR CHOICE: a value of x outside your data, for question 3
# (use a decimal point here: 25.5)
x_outside = 60

# What your line says there
outside = predict(x_outside, w_mine, b_mine)
# Show it
print(f"at {x_outside:g} {x_unit}, your line says {outside:.4g} {y_unit}")
# For comparison: your data's range (x_min, x_max from the cell above)
print(f"(your data goes from {x_min:g} to {x_max:g} {x_unit})")

### Your answers

One to three sentences per question, about your own example.

1. **What does w say?** Put it in everyday words, like "every extra kilometre costs me about four
   minutes". Does that sound right to you?
2. **Your case and the line.** Is your case above, below or on the line, and by how much? Can you
   think of a reason, or can you not tell from your data?
3. **Where would you not trust it?** The last cell shows what the line says outside your data.
   Would you believe that number? Why or why not?
4. **What does your line not know?** In class, the line did not know the U-value of the wall. What
   might matter for your two quantities that is not in your data?

<details>
<summary>Example answers for the WG-room example. Click to see what a good answer looks like.</summary>

1. Every extra square metre goes with about 16 € more rent per month. That sounds right to
   me: in listings, a big room clearly costs more than a small one.
2. My room is 73 € above the line, almost three times the typical miss of 26 €. It
   has a balcony and is two minutes from the U-Bahn, and the line knows neither.
3. For a 60 m² room the line says 1186 €. My data only goes from 10 to 26 m², and a room of
   60 m² is a flat rather than a WG room, so I would not believe it.
4. Location, a balcony, whether heating is included, the age of the building. None of that is in my
   two columns, and all of it changes the rent.

These fit the example only. Your answers are about your own pairs.

</details>

*Your text here.*